# DeepSeek-V2-Lite end-to-end EPCB benchmark (A100)

Validates whether the per-layer cache **structural** claim (+14.7pp on DeepSeek, currently trace-replay only) translates to wall-clock speedup on A100 — closing the §5.6 narrative gap from the paper.

**Hardware required:** Colab Pro or Pro+ with **A100** (40 or 80 GB). Check below; if you get a T4 or V100, change runtime type.

**Time:** ~10 min one-time model download (~32 GB → Drive cache, persists across sessions), then ~25–35 min for the benchmark.

**Configurations compared:**
1. `baseline_auto` — `device_map='auto'`, reference point.
2. `skeleton_cap1` — expert-aware loading + capacity-1 nocache (isolates loading).
3. `flat_cap32` — shared cache, cap=32 LFU+history.
4. `uniform_b864` — per-layer caches, 32/layer × 27 = 864 total, uniform.
5. `entropy_b864` — same total, Shannon-entropy-allocated.

**Decision rule for the paper:**
- If `uniform_b864 > flat_cap32` in tok/s → the per-layer structural claim is validated end-to-end → EPCB content stays / comes back to main.tex.
- If `entropy_b864 > uniform_b864` → the entropy signal also validates end-to-end.
- If both flat → EPCB stays cut; the focused-paper refactor was correct.

## 1. Verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
assert 'A100' in gpu, f'Need A100, got {gpu}. Runtime > Change runtime type > A100.'

## 2. Mount Drive and point HF cache at it

DeepSeek-V2-Lite is ~32 GB. Caching it to Drive means subsequent runs start instantly instead of re-downloading.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
os.environ['HF_HUB_CACHE'] = '/content/drive/MyDrive/hf_cache/hub'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('HF cache:', os.environ['HF_HOME'])
!du -sh /content/drive/MyDrive/hf_cache 2>/dev/null || echo '(empty, will populate on first download)'

## 3. Install dependencies and clone the repo

We clone the repo for the `moe_policylang` package source (PyPI version may lag the local `load_moe_model` improvements). If you prefer PyPI, change the clone cell to `!pip install -q moe-policylang`.

In [ ]:
!pip install -q transformers accelerate psutil lark
!git clone -q https://github.com/jesse-pokora/MoE-PolicyLang.git /content/repo
import sys
sys.path.insert(0, '/content/repo')
import moe_policylang
print('moe_policylang loaded from', moe_policylang.__file__)

## 4. Write the benchmark script

Embedded here so the notebook is self-contained — if the script in the cloned repo is older than this notebook, this cell wins.

In [ ]:
%%writefile /content/bench_deepseek_a100.py
#!/usr/bin/env python3
"""End-to-end EPCB validation on DeepSeek-V2-Lite (A100, fp16)."""
import argparse, gc, json, os, statistics, sys, time

sys.path.insert(0, '/content/repo')

import torch
import psutil

MODEL_ID = "deepseek-ai/DeepSeek-V2-Lite"
PROMPT = "Explain the key ideas behind mixture-of-experts models."
OUT_PATH = "/content/drive/MyDrive/deepseek_a100_results.json"


def attach_policy(model, dsl):
    from moe_policylang.parser import parse_policy
    from moe_policylang.compiler import compile_policy
    from moe_policylang.runtime.hooks import build_hook
    from moe_policylang.integrations.accessors import auto_accessor
    from moe_policylang.integrations.weight_placement import WeightPlacementManager
    accessor = auto_accessor(model)
    ir = parse_policy(dsl)
    compiled = compile_policy(ir)
    hook = build_hook(compiled, num_layers=accessor.num_layers, num_experts=accessor.num_experts)
    mgr = WeightPlacementManager(hook, accessor)
    mgr.attach()
    return mgr


def reset_stats(mgr):
    hook = mgr.hook
    if hasattr(hook, '_per_layer_hooks'):
        for h in hook._per_layer_hooks.values():
            h.cache.stats.hits = 0; h.cache.stats.misses = 0; h.cache.stats.evictions = 0
    elif hasattr(hook, 'inner') and hasattr(hook.inner, 'cache'):
        hook.inner.cache.stats.hits = 0; hook.inner.cache.stats.misses = 0; hook.inner.cache.stats.evictions = 0
    elif hasattr(hook, 'cache'):
        hook.cache.stats.hits = 0; hook.cache.stats.misses = 0; hook.cache.stats.evictions = 0
    if hasattr(mgr, 'stats'):
        mgr.stats.cpu_to_gpu_transfers = 0
        mgr.stats.gpu_to_cpu_transfers = 0
        mgr.stats.bytes_transferred = 0
        mgr.stats.transfer_time_s = 0.0


def run_once(model, tok, mgr, max_tokens, prompt):
    if mgr is not None:
        reset_stats(mgr)
    inp = tok(prompt, return_tensors='pt').to(model.device)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    gen = out.shape[1] - inp['input_ids'].shape[1]
    gpu_gb = torch.cuda.memory_allocated() / 1e9
    if mgr is None:
        return gen / elapsed, gpu_gb, None, 0
    stats = mgr.get_stats()
    return gen / elapsed, gpu_gb, stats['policy']['cache']['hit_rate'], stats['placement']['cpu_to_gpu_transfers']


def measure(model, tok, mgr, label, max_tokens, warmup, runs):
    print(f'\n--- {label} ---')
    tps, gpu, hr, xf = [], [], [], []
    for w in range(warmup):
        t, g, h, x = run_once(model, tok, mgr, max_tokens, PROMPT)
        print(f'  warmup {w+1}: {t:.2f} tok/s  GPU={g:.1f}GB  hit={(h or 0)*100:.1f}%')
    for r in range(runs):
        t, g, h, x = run_once(model, tok, mgr, max_tokens, PROMPT)
        tps.append(t); gpu.append(g)
        if h is not None: hr.append(h); xf.append(x)
        print(f'  run {r+1}/{runs}: {t:.2f} tok/s  GPU={g:.1f}GB  hit={(h or 0)*100:.1f}%  xfer={x}')
    return {
        'tps_values': tps,
        'tps_mean': statistics.mean(tps),
        'tps_std': statistics.stdev(tps) if len(tps) > 1 else 0.0,
        'tps_steady_mean': statistics.mean(tps[1:]) if len(tps) >= 2 else tps[0],
        'tps_steady_std': statistics.stdev(tps[1:]) if len(tps) >= 3 else 0.0,
        'gpu_gb_mean': statistics.mean(gpu),
        'hit_rate_mean': statistics.mean(hr) if hr else None,
        'transfers_mean': statistics.mean(xf) if xf else 0,
    }


def per_layer_dsl(mode, total_budget, num_layers, num_experts):
    avg = total_budget // num_layers
    if mode == 'uniform':
        return (
            f'policy uniform_b{total_budget} {{ '
            f'cache {{ capacity = {avg}  eviction = lfu  frequency_decay = 0.9 }} '
            f'prefetch {{ strategy = history  budget = 2 }} '
            f'per_layer {{ allocation = entropy  entropy_window = 200  '
            f'min_capacity = {avg}  max_capacity = {avg}  rebalance_interval = 999999  total_budget = {total_budget} }} }}'
        )
    elif mode == 'entropy':
        lo = max(2, avg // 2); hi = min(num_experts, avg * 3)
        return (
            f'policy entropy_b{total_budget} {{ '
            f'cache {{ capacity = {avg}  eviction = lfu  frequency_decay = 0.9 }} '
            f'prefetch {{ strategy = history  budget = 2 }} '
            f'per_layer {{ allocation = entropy  entropy_window = 200  '
            f'min_capacity = {lo}  max_capacity = {hi}  rebalance_interval = 500  total_budget = {total_budget} }} }}'
        )
    raise ValueError(mode)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--runs', '-n', type=int, default=5)
    ap.add_argument('--warmup', type=int, default=1)
    ap.add_argument('--max-tokens', type=int, default=64)
    ap.add_argument('--skip-baseline', action='store_true')
    ap.add_argument('--budget-large', action='store_true')
    args = ap.parse_args()

    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}  VRAM: {vram_gb:.1f} GB')
    print(f'RAM: {psutil.virtual_memory().total/1e9:.1f} GB total')
    print(f'HF cache: {os.environ.get("HF_HOME")}')

    results = {'model': MODEL_ID, 'gpu': gpu_name, 'vram_gb': vram_gb,
               'runs': args.runs, 'warmup': args.warmup, 'max_tokens': args.max_tokens,
               'configs': {}}

    if not args.skip_baseline:
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            print('\nLoading baseline (device_map=auto)...')
            t0 = time.perf_counter()
            bm = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16,
                                                      device_map='auto', trust_remote_code=True)
            bm.eval()
            bt = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
            if bt.pad_token is None: bt.pad_token = bt.eos_token
            print(f'  loaded in {time.perf_counter()-t0:.1f}s, GPU={torch.cuda.memory_allocated()/1e9:.1f}GB')
            results['configs']['baseline_auto'] = measure(bm, bt, None, 'baseline_auto',
                                                          args.max_tokens, args.warmup, args.runs)
            del bm, bt
            gc.collect(); torch.cuda.empty_cache()
        except Exception as e:
            print(f'  baseline failed: {e}')
            results['configs']['baseline_auto'] = {'error': str(e)}
            gc.collect(); torch.cuda.empty_cache()

    import moe_policylang
    print('\nLoading with expert-aware device map...')
    t0 = time.perf_counter()
    model, tok = moe_policylang.load_moe_model(MODEL_ID, trust_remote_code=True)
    print(f'  loaded in {time.perf_counter()-t0:.1f}s, skeleton {torch.cuda.memory_allocated()/1e9:.2f}GB')

    from moe_policylang.integrations.accessors import auto_accessor
    a = auto_accessor(model)
    results['num_layers'] = a.num_layers
    results['num_experts'] = a.num_experts
    results['skeleton_gpu_gb'] = torch.cuda.memory_allocated() / 1e9
    print(f'  {a.num_layers} layers, {a.num_experts} experts')

    mgr = attach_policy(model, 'policy skeleton { cache { capacity = 1  eviction = lru } prefetch { budget = 1 } }')
    results['configs']['skeleton_cap1'] = measure(model, tok, mgr, 'skeleton_cap1', args.max_tokens, args.warmup, args.runs)
    mgr.detach(); gc.collect(); torch.cuda.empty_cache()

    mgr = attach_policy(model,
        'policy flat_cap32 { '
        'cache { capacity = 32  eviction = lfu  frequency_decay = 0.9 } '
        'prefetch { strategy = history  budget = 4 } }')
    results['configs']['flat_cap32'] = measure(model, tok, mgr, 'flat_cap32', args.max_tokens, args.warmup, args.runs)
    mgr.detach(); gc.collect(); torch.cuda.empty_cache()

    BUDGETS = [864]
    if args.budget_large: BUDGETS.append(1296)
    for budget in BUDGETS:
        for mode in ('uniform', 'entropy'):
            name = f'{mode}_b{budget}'
            mgr = attach_policy(model, per_layer_dsl(mode, budget, a.num_layers, a.num_experts))
            results['configs'][name] = measure(model, tok, mgr, f'{name} ({budget // a.num_layers}/layer)',
                                                args.max_tokens, args.warmup, args.runs)
            mgr.detach(); gc.collect(); torch.cuda.empty_cache()

    print('\n' + '=' * 80)
    print('SUMMARY')
    print('=' * 80)
    print(f'{"Config":<22s}  {"steady tok/s":<18s}  {"GPU(GB)":<8s}  {"Hit%":<7s}  {"xfer":<6s}')
    print('-' * 80)
    for name, c in results['configs'].items():
        if 'error' in c:
            print(f'{name:<22s}  ERROR: {c["error"]}')
            continue
        st = f'{c["tps_steady_mean"]:.2f} +/- {c["tps_steady_std"]:.2f}'
        hr = f'{c["hit_rate_mean"]*100:5.1f}%' if c.get('hit_rate_mean') is not None else '  ---'
        print(f'{name:<22s}  {st:<18s}  {c["gpu_gb_mean"]:<8.1f}  {hr:<7s}  {int(c.get("transfers_mean") or 0):<6d}')

    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    with open(OUT_PATH, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'\nSaved to {OUT_PATH}')


if __name__ == '__main__':
    main()


## 5. Run the benchmark

First run downloads ~32 GB of weights from HuggingFace. After that, the benchmark itself runs ~25–35 min.

Output is appended to `/content/drive/MyDrive/deepseek_a100_results.json` so it survives runtime resets.

In [ ]:
# Run the benchmark. Output is shown live AND copied to /content/bench.log
# so cell 12 can tail it if you re-run diagnostics partway.
!python /content/bench_deepseek_a100.py --runs 5 --max-tokens 64 2>&1 | tee /content/bench.log

## 6. Inspect results

Compute the steady-state means and the key comparisons that will (or won't) bring EPCB back into the paper.

In [ ]:
import json, os, subprocess

RESULTS_PATH = '/content/drive/MyDrive/deepseek_a100_results.json'

if not os.path.exists(RESULTS_PATH):
    print(f'Results file not found: {RESULTS_PATH}')
    print()
    print('The benchmark (cell 10) has not completed yet. Diagnostics below.')
    print()

    cache_path = '/content/drive/MyDrive/hf_cache'
    if os.path.exists(cache_path):
        print('--- HF cache state ---')
        r = subprocess.run(
            f'du -sh {cache_path}/hub/models--* 2>/dev/null',
            shell=True, capture_output=True, text=True,
        )
        print(r.stdout if r.stdout.strip() else '(empty — model has not started downloading)')

        # Also check if there are .incomplete files (active downloads)
        r2 = subprocess.run(
            f'find {cache_path}/hub -name "*.incomplete" 2>/dev/null | wc -l',
            shell=True, capture_output=True, text=True,
        )
        in_flight = int(r2.stdout.strip() or 0)
        if in_flight > 0:
            print(f'\n{in_flight} file(s) still downloading (.incomplete present).')
    else:
        print(f'HF cache directory does not exist: {cache_path}')

    # Tail the bench script's last output if redirected to a log
    log_path = '/content/bench.log'
    if os.path.exists(log_path):
        print('\n--- Last 30 lines of /content/bench.log ---')
        r3 = subprocess.run(f'tail -30 {log_path}', shell=True, capture_output=True, text=True)
        print(r3.stdout)
    else:
        print('\nNo /content/bench.log yet. The bench cell is either still running or')
        print('errored before producing any output. Scroll up to cell 10 output for the')
        print('Traceback or current progress.')
    print('\nTo proceed: wait for cell 10 to finish (it prints a SUMMARY block at the end),')
    print('or interrupt and report the last visible output in cell 10.')
else:
    with open(RESULTS_PATH) as f:
        d = json.load(f)

    print(f'Model: {d["model"]}')
    print(f'GPU: {d["gpu"]}  VRAM: {d["vram_gb"]:.1f} GB')
    print(f'Layers: {d.get("num_layers")}, experts: {d.get("num_experts")}')
    print(f'Skeleton on GPU: {d.get("skeleton_gpu_gb", 0):.2f} GB')
    print()
    print(f'{"Config":<22s}  {"tok/s (steady)":<18s}  {"GPU(GB)":<8s}  {"Hit%":<7s}')
    print('-' * 70)
    for name, c in d['configs'].items():
        if 'error' in c:
            print(f'{name:<22s}  ERROR: {c["error"]}'); continue
        st = f'{c["tps_steady_mean"]:.2f} +/- {c["tps_steady_std"]:.2f}'
        hr = f'{c["hit_rate_mean"]*100:5.1f}%' if c.get('hit_rate_mean') is not None else '  ---'
        print(f'{name:<22s}  {st:<18s}  {c["gpu_gb_mean"]:<8.1f}  {hr:<7s}')

    print('\n--- Decision rule ---')
    configs = d['configs']
    if 'flat_cap32' in configs and 'uniform_b864' in configs and 'tps_steady_mean' in configs['flat_cap32']:
        delta_struct = configs['uniform_b864']['tps_steady_mean'] - configs['flat_cap32']['tps_steady_mean']
        pct = 100 * delta_struct / configs['flat_cap32']['tps_steady_mean']
        print(f'Per-layer (uniform_b864) vs flat_cap32: {delta_struct:+.2f} tok/s ({pct:+.1f}%)')
        if delta_struct > 0.5:
            print('  -> structural per-layer claim VALIDATED end-to-end. Restore EPCB to paper.')
        elif delta_struct > -0.5:
            print('  -> within noise. EPCB stays cut.')
        else:
            print('  -> per-layer HURTS on A100 too. Keep EPCB cut, possibly reconsider claim.')
    if 'uniform_b864' in configs and 'entropy_b864' in configs:
        delta_entropy = configs['entropy_b864']['tps_steady_mean'] - configs['uniform_b864']['tps_steady_mean']
        pct = 100 * delta_entropy / configs['uniform_b864']['tps_steady_mean']
        print(f'Entropy (entropy_b864) vs uniform_b864: {delta_entropy:+.2f} tok/s ({pct:+.1f}%)')
        if delta_entropy > 0.3:
            print('  -> entropy signal validated end-to-end.')
        else:
            print('  -> entropy signal does not differentiate in wall-clock.')

## Notes

- **If `baseline_auto` fails** with OOM or hangs: pass `--skip-baseline` to the run cell. The reference point is then implicit (we know from the original paper that `device_map='auto'` on a 32 GB model with a 40 GB A100 should *just* fit, but it's borderline).
- **If `load_moe_model` fails on shared experts**: DeepSeek-V2-Lite has shared experts (always-on, not routed). If the auto-accessor mishandles these, you'll see an error at the skeleton load step. Report the traceback and I can patch the accessor.
- **Tighter budgets**: change `BUDGETS = [864]` in the script to e.g. `[432, 864, 1296]` (16, 32, 48 per layer) for a sweep matching paper Table 3.
- **Larger budgets**: pass `--budget-large` to add 1296 (48/layer) to the sweep.